In [1]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
print(tf.__version__)

I0000 00:00:1795546222.937979 1943437 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2.21.0


In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2026-11-24 21:50:25--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
cdn.freecodecamp.org (cdn.freecodecamp.org) çözümleniyor... 2606:4700:20::681a:221, 2606:4700:20::681a:321, 2606:4700:20::ac43:4695, ...
cdn.freecodecamp.org (cdn.freecodecamp.org)|2606:4700:20::681a:221|:443 bağlanılıyor... 

bağlantı kuruldu.


HTTP isteği gönderildi, cevap bekleniyor... 

200 OK
Uzunluk: 358233 (350K) [text/tab-separated-values]
Kayıt yeri: ‘train-data.tsv.3’

train-data.tsv.3      0%[                    ]       0  --.-KB/s               

train-data.tsv.3    100%[===================>] 349,84K  1,60MB/s    süre 0,2s  

2026-11-24 21:50:26 (1,60 MB/s) - ‘train-data.tsv.3’ kaydedildi [358233/358233]



--2026-11-24 21:50:26--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
cdn.freecodecamp.org (cdn.freecodecamp.org) çözümleniyor... 2606:4700:20::ac43:4695, 2606:4700:20::681a:321, 2606:4700:20::681a:221, ...
cdn.freecodecamp.org (cdn.freecodecamp.org)|2606:4700:20::ac43:4695|:443 bağlanılıyor... 

bağlantı kuruldu.


HTTP isteği gönderildi, cevap bekleniyor... 

200 OK
Uzunluk: 118774 (116K) [text/tab-separated-values]
Kayıt yeri: ‘valid-data.tsv.3’

valid-data.tsv.3      0%[                    ]       0  --.-KB/s               

valid-data.tsv.3    100%[===================>] 115,99K  --.-KB/s    süre 0,06s 

2026-11-24 21:50:26 (2,01 MB/s) - ‘valid-data.tsv.3’ kaydedildi [118774/118774]



In [3]:
train = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'msg'])
test = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'msg'])
y_train = (train['label'] == 'spam').astype(int).values
y_test = (test['label'] == 'spam').astype(int).values
vec = keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=60)
X_train = tf.constant(train['msg'].astype(str).tolist()); X_test = tf.constant(test['msg'].astype(str).tolist())
vec.adapt(X_train)

In [4]:
model = keras.Sequential([keras.layers.Input(shape=(1,), dtype=tf.string), vec,
    keras.layers.Embedding(10000, 16), keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(16, activation='relu'), keras.layers.Dense(1, activation='sigmoid')])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=15, validation_data=(X_test, y_test), verbose=2, class_weight={0: 1.0, 1: 3.0})

Epoch 1/15


131/131 - 1s - 10ms/step - accuracy: 0.8516 - loss: 0.7669 - val_accuracy: 0.8657 - val_loss: 0.4265


Epoch 2/15


131/131 - 0s - 3ms/step - accuracy: 0.8660 - loss: 0.6901 - val_accuracy: 0.8649 - val_loss: 0.3674


Epoch 3/15


131/131 - 0s - 2ms/step - accuracy: 0.9064 - loss: 0.5354 - val_accuracy: 0.9598 - val_loss: 0.2750


... (20 outputs left out)

Epoch 14/15


131/131 - 0s - 3ms/step - accuracy: 0.9943 - loss: 0.0466 - val_accuracy: 0.9828 - val_loss: 0.0644


Epoch 15/15


131/131 - 0s - 2ms/step - accuracy: 0.9952 - loss: 0.0436 - val_accuracy: 0.9828 - val_loss: 0.0531


In [5]:
def predict_message(pred_text):
  p = float(model.predict(tf.constant([pred_text]), verbose=0)[0][0])
  prediction = [p, 'spam' if p > 0.5 else 'ham']
  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

[0.005633979570120573, 'ham']


In [6]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


You passed the challenge. Great job!
